# Phase 4 – Feature Engineering

The objective of this phase is to transform the cleaned datasets into a machine learning-ready dataset by creating meaningful features that improve demand forecasting and inventory risk prediction.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Load Cleaned Datasets

In this section, the cleaned datasets created during Phase 2 are loaded into the notebook. These datasets will be used to create machine learning features for demand forecasting and inventory intelligence.

In [2]:
customer_df = pd.read_csv("../data/processed/customer_master_clean.csv")

inventory_df = pd.read_csv("../data/processed/inventory_snapshot_clean.csv")

promotions_df = pd.read_csv("../data/processed/promotions_clean.csv")

sales_df = pd.read_csv("../data/processed/sales_transactions_clean.csv")

sku_flags_df = pd.read_csv("../data/processed/sku_inventory_flags_clean.csv")

sku_df = pd.read_csv("../data/processed/sku_master_clean.csv")

store_df = pd.read_csv("../data/processed/store_master_clean.csv")

In [3]:
print("Customer:", customer_df.shape)
print("Inventory:", inventory_df.shape)
print("Promotions:", promotions_df.shape)
print("Sales:", sales_df.shape)
print("SKU Inventory Flags:", sku_flags_df.shape)
print("SKU Master:", sku_df.shape)
print("Store:", store_df.shape)

Customer: (10000, 7)
Inventory: (26408, 6)
Promotions: (100, 8)
Sales: (9945511, 11)
SKU Inventory Flags: (600, 6)
SKU Master: (5000, 7)
Store: (30, 5)


## Dataset Overview

The sales dataset is the primary dataset for forecasting. This section verifies the structure, data types, and sample records before creating new features.

In [4]:
sales_df.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN


In [5]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          str    
 1   receipt_id    str    
 2   store_id      str    
 3   sku_id        str    
 4   customer_id   str    
 5   quantity      int64  
 6   unit_price    float64
 7   total_value   float64
 8   channel       str    
 9   discount_pct  float64
 10  promo_id      str    
dtypes: float64(3), int64(1), str(7)
memory usage: 834.7 MB


In [6]:
sales_df["date"] = pd.to_datetime(sales_df["date"])

In [7]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype         
---  ------        -----         
 0   date          datetime64[us]
 1   receipt_id    str           
 2   store_id      str           
 3   sku_id        str           
 4   customer_id   str           
 5   quantity      int64         
 6   unit_price    float64       
 7   total_value   float64       
 8   channel       str           
 9   discount_pct  float64       
 10  promo_id      str           
dtypes: datetime64[us](1), float64(3), int64(1), str(6)
memory usage: 834.7 MB


In [8]:
sales_df.isnull().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7855345
dtype: int64

In [9]:
sales_df.duplicated().sum()

np.int64(12897)

In [10]:
sales_df = sales_df.drop_duplicates()

print("Duplicate rows removed successfully.")

Duplicate rows removed successfully.


In [11]:
sales_df.duplicated().sum()

np.int64(0)

In [12]:
sales_df.shape

(9932614, 11)

### Duplicate Handling

A total of **12,897 duplicate rows** were identified in the sales dataset. These duplicate records were removed to ensure that each transaction is unique and to prevent inflated sales values during feature engineering and model training.

## Date Feature Engineering

In this section, new time-based features are extracted from the transaction date. These features help the forecasting model identify trends, seasonality, and weekly or monthly purchasing patterns.

In [13]:
# Year
sales_df["year"] = sales_df["date"].dt.year

# Month
sales_df["month"] = sales_df["date"].dt.month

# Quarter
sales_df["quarter"] = sales_df["date"].dt.quarter

# Week Number
sales_df["week"] = sales_df["date"].dt.isocalendar().week.astype(int)

# Day of Week
sales_df["day_of_week"] = sales_df["date"].dt.day_name()

# Weekend Indicator
sales_df["is_weekend"] = sales_df["day_of_week"].isin(["Saturday", "Sunday"])

In [14]:
sales_df[
    [
        "date",
        "year",
        "month",
        "quarter",
        "week",
        "day_of_week",
        "is_weekend"
    ]
].head()

,date,year,month,quarter,week,day_of_week,is_weekend
0,2025-04-02,2025,4,2,14,Wednesday,False
1,2025-04-02,2025,4,2,14,Wednesday,False
2,2022-04-24,2022,4,2,16,Sunday,True
3,2022-04-24,2022,4,2,16,Sunday,True
4,2024-09-22,2024,9,3,38,Sunday,True


In [15]:
sales_df.info()

<class 'pandas.DataFrame'>
Index: 9932614 entries, 0 to 9945510
Data columns (total 17 columns):
 #   Column        Dtype         
---  ------        -----         
 0   date          datetime64[us]
 1   receipt_id    str           
 2   store_id      str           
 3   sku_id        str           
 4   customer_id   str           
 5   quantity      int64         
 6   unit_price    float64       
 7   total_value   float64       
 8   channel       str           
 9   discount_pct  float64       
 10  promo_id      str           
 11  year          int32         
 12  month         int32         
 13  quarter       int32         
 14  week          int64         
 15  day_of_week   str           
 16  is_weekend    bool          
dtypes: bool(1), datetime64[us](1), float64(3), int32(3), int64(2), str(7)
memory usage: 1.2 GB


### Date Feature Engineering Summary

The transaction date was transformed into multiple time-based features, including year, month, quarter, week number, day of the week, and a weekend indicator. These engineered features capture temporal patterns and seasonality, improving the predictive capability of the demand forecasting model.

## SKU-Level Sales Features

In this section, transaction-level sales data is aggregated at the SKU level to create features that describe product performance. These features provide information about sales volume, revenue, pricing, and transaction frequency.

In [16]:
sku_features = (
    sales_df
    .groupby("sku_id")
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("total_value", "sum"),
        average_price=("unit_price", "mean"),
        total_transactions=("receipt_id", "count")
    )
    .reset_index()
)

In [17]:
sku_features.head()

,sku_id,total_quantity,total_sales,average_price,total_transactions
0,SKU00001,4771,3624393.59,813.41,2529
1,SKU00002,38154,2517245.14,70.38,20401
2,SKU00003,1325,189058.31,151.28,713
3,SKU00004,2534,556401.40,233.64,1373
4,SKU00005,2292,300920.55,138.50,1219


In [18]:
sku_features.shape

(5000, 5)

In [19]:
sku_features = sku_features.merge(
    sku_df,
    on="sku_id",
    how="left"
)

In [20]:
sku_features.head()

,sku_id,total_quantity,total_sales,average_price,total_transactions,sku_name,category,subcategory,unit_price,cost_price,brand
0,SKU00001,4771,3624393.59,813.41,2529,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,38154,2517245.14,70.38,20401,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,1325,189058.31,151.28,713,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,SKU00004,2534,556401.40,233.64,1373,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,SKU00005,2292,300920.55,138.50,1219,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [21]:
sku_features.shape

(5000, 11)

In [22]:
sku_features.isnull().sum()

sku_id                0
total_quantity        0
total_sales           0
average_price         0
total_transactions    0
sku_name              0
category              0
subcategory           0
unit_price            0
cost_price            0
brand                 0
dtype: int64

### SKU-Level Feature Engineering Summary

Transaction-level sales data was aggregated to generate SKU-level features, including total quantity sold, total sales revenue, average selling price, and transaction count. These features were then combined with product master information such as category, subcategory, brand, and pricing details, resulting in a comprehensive dataset for forecasting and inventory analysis.

## Customer-Level Features

Customer-level features are created by aggregating transaction data for each customer. These features summarize purchasing behavior and help identify valuable customer segments.

In [23]:
customer_features = (
    sales_df
    .groupby("customer_id")
    .agg(
        total_spent=("total_value", "sum"),
        total_quantity=("quantity", "sum"),
        total_transactions=("receipt_id", "count"),
        average_transaction_value=("total_value", "mean")
    )
    .reset_index()
)

In [24]:
customer_features.head()

,customer_id,total_spent,total_quantity,total_transactions,average_transaction_value
0,CUST00001,943800.74,1626,878,1074.943895
1,CUST00002,965918.81,1674,879,1098.883743
2,CUST00003,649917.08,1164,619,1049.946817
3,CUST00004,1028273.13,1710,902,1139.992384
4,CUST00005,598947.09,1070,565,1060.083345


In [25]:
customer_features.shape

(10000, 5)

In [26]:
customer_features = customer_features.merge(
    customer_df,
    left_on="customer_id",
    right_on="cust_id",
    how="left"
)

In [27]:
customer_features.head()

,customer_id,total_spent,total_quantity,total_transactions,average_transaction_value,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date
0,CUST00001,943800.74,1626,878,1074.943895,CUST00001,52,Female,Sukkur,Silver,Online,2021-01-01
1,CUST00002,965918.81,1674,879,1098.883743,CUST00002,41,Female,Karachi,Silver,Mobile App,2021-05-14
2,CUST00003,649917.08,1164,619,1049.946817,CUST00003,48,Female,Bahawalpur,Bronze,In-Store,2022-10-23
3,CUST00004,1028273.13,1710,902,1139.992384,CUST00004,45,Male,Karachi,Silver,In-Store,2015-07-15
4,CUST00005,598947.09,1070,565,1060.083345,CUST00005,39,Female,Faisalabad,Bronze,In-Store,2024-01-02


In [28]:
customer_features.isnull().sum()

customer_id                  0
total_spent                  0
total_quantity               0
total_transactions           0
average_transaction_value    0
cust_id                      0
age                          0
gender                       0
city                         0
loyalty_segment              0
preferred_channel            0
registration_date            0
dtype: int64

In [29]:
customer_features.shape

(10000, 12)

### Customer-Level Feature Engineering Summary

Customer transaction data was aggregated to create behavioral features such as total spending, purchase quantity, transaction count, and average transaction value. These features were enriched with customer demographic information, providing a comprehensive view of customer purchasing patterns for downstream analytics.

## Store-Level Features

Store-level features are created by aggregating transaction data for each store. These features summarize store performance and are enriched with store metadata such as location and store type.

In [30]:
store_features = (
    sales_df
    .groupby("store_id")
    .agg(
        total_sales=("total_value", "sum"),
        total_quantity=("quantity", "sum"),
        total_transactions=("receipt_id", "count"),
        average_order_value=("total_value", "mean")
    )
    .reset_index()
)

In [31]:
store_features.head()

,store_id,total_sales,total_quantity,total_transactions,average_order_value
0,ST01,2.578563e+08,443317,235971,1092.745882
1,ST02,1.778035e+08,304054,161731,1099.377814
2,ST03,3.851741e+08,662395,352307,1093.291079
3,ST04,3.873425e+08,663770,352675,1098.298812
4,ST05,3.887403e+08,665465,354009,1098.108522


In [32]:
store_features.shape

(30, 5)

In [33]:
store_features = store_features.merge(
    store_df,
    on="store_id",
    how="left"
)

In [34]:
store_features.head()

,store_id,total_sales,total_quantity,total_transactions,average_order_value,store_name,city,store_type,opening_date
0,ST01,2.578563e+08,443317,235971,1092.745882,Quetta Convenience Store #01,Quetta,Convenience Store,2017-07-08
1,ST02,1.778035e+08,304054,161731,1099.377814,Islamabad Express Store #02,Islamabad,Express Store,2019-05-08
2,ST03,3.851741e+08,662395,352307,1093.291079,Sialkot Supermarket #03,Sialkot,Supermarket,2019-03-28
3,ST04,3.873425e+08,663770,352675,1098.298812,Multan Supermarket #04,Multan,Supermarket,2017-10-08
4,ST05,3.887403e+08,665465,354009,1098.108522,Karachi Supermarket #05,Karachi,Supermarket,2020-11-03


In [35]:
store_features.isnull().sum()

store_id               0
total_sales            0
total_quantity         0
total_transactions     0
average_order_value    0
store_name             0
city                   0
store_type             0
opening_date           0
dtype: int64

In [36]:
store_features.shape

(30, 9)

### Store-Level Feature Engineering Summary

Store transaction data was aggregated to generate performance metrics such as total sales, total quantity sold, transaction count, and average order value. These features were combined with store metadata including city, store type, and opening date to create a comprehensive store-level dataset for operational analysis.

## Promotion Features

Promotion-related features are created by combining sales transactions with promotion information. These features help the forecasting model understand how promotional campaigns influence customer demand.

In [37]:
sales_with_promotions = sales_df.merge(
    promotions_df,
    on="promo_id",
    how="left"
)

In [38]:
sales_with_promotions.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct_x,...,week,day_of_week,is_weekend,promo_name,start_date,end_date,discount_pct_y,promo_type,target_type,target_value
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,...,14,Wednesday,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,...,14,Wednesday,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,...,16,Sunday,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,...,16,Sunday,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,...,38,Sunday,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
sales_with_promotions.shape

(9932614, 24)

In [40]:
sales_with_promotions.isnull().sum()

date                    0
receipt_id              0
store_id                0
sku_id                  0
customer_id             0
quantity                0
unit_price              0
total_value             0
channel                 0
discount_pct_x          0
promo_id          7845164
year                    0
month                   0
quarter                 0
week                    0
day_of_week             0
is_weekend              0
promo_name        7845164
start_date        7845164
end_date          7845164
discount_pct_y    7845164
promo_type        7845164
target_type       7845164
target_value      7845164
dtype: int64

In [41]:
sales_with_promotions["promotion_used"] = sales_with_promotions["promo_id"].notna()

In [42]:
sales_with_promotions[
    ["promo_id", "promotion_used"]
].sample(10, random_state=42)

,promo_id,promotion_used
3233927,PROMO075,True
5017987,PROMO043,True
3199494,NaN,False
3720497,NaN,False
435921,NaN,False
5675479,NaN,False
7255367,NaN,False
7533124,NaN,False
7018896,NaN,False
4303596,NaN,False


In [43]:
sales_with_promotions["promotion_used"].value_counts()

promotion_used
False    7845164
True     2087450
Name: count, dtype: int64

### Promotion Feature Engineering Summary

Sales transactions were merged with the promotions dataset using the `promo_id` field. Since most transactions were not associated with promotional campaigns, missing values in the promotion-related columns are expected. A new boolean feature, `promotion_used`, was created to indicate whether each transaction occurred under an active promotion. This feature will help the forecasting model learn the effect of promotions on product demand.

## Inventory Feature Engineering

Inventory-related features are created by combining inventory snapshot data with product information. These features help identify inventory availability, reorder requirements, and potential stock risks for each SKU.

In [44]:
inventory_df.head()

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date
0,ST11,SKU02558,333,72,22,2025-06-23
1,ST21,SKU01031,236,67,14,2025-06-17
2,ST26,SKU02129,496,96,34,2025-07-10
3,ST19,SKU02907,109,34,13,2025-08-17
4,ST05,SKU01023,333,97,22,2025-07-12


In [45]:
inventory_df.shape

(26408, 6)

In [46]:
inventory_features = inventory_df.merge(
    sku_df,
    on="sku_id",
    how="left"
)

In [47]:
inventory_features.head()

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date,sku_name,category,subcategory,unit_price,cost_price,brand
0,ST11,SKU02558,333,72,22,2025-06-23,PrimeCare Butter & Ghee 2L,Dairy & Bakery,Butter & Ghee,87.70,67.29,PrimeCare
1,ST21,SKU01031,236,67,14,2025-06-17,PrimeCare First Aid 500g,Health & Wellness,First Aid,668.10,419.47,PrimeCare
2,ST26,SKU02129,496,96,34,2025-07-10,UrbanBite Oral Care 500g,Personal Care,Oral Care,227.28,138.59,UrbanBite
3,ST19,SKU02907,109,34,13,2025-08-17,NovaFresh Kids Wear Large,Apparel & Footwear,Kids Wear,1495.16,1152.38,NovaFresh
4,ST05,SKU01023,333,97,22,2025-07-12,CrispKing Men's Grooming Large,Personal Care,Men's Grooming,526.16,382.68,CrispKing


In [48]:
inventory_features.isnull().sum()

store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
sku_name             0
category             0
subcategory          0
unit_price           0
cost_price           0
brand                0
dtype: int64

In [49]:
inventory_features["below_reorder_point"] = (
    inventory_features["stock_on_hand"] <
    inventory_features["reorder_point"]
)

In [50]:
inventory_features["below_safety_stock"] = (
    inventory_features["stock_on_hand"] <
    inventory_features["safety_stock"]
)

In [51]:
inventory_features["stock_ratio"] = (
    inventory_features["stock_on_hand"] /
    inventory_features["reorder_point"]
)

In [52]:
inventory_features[
    [
        "stock_on_hand",
        "reorder_point",
        "safety_stock",
        "below_reorder_point",
        "below_safety_stock",
        "stock_ratio"
    ]
].head()

,stock_on_hand,reorder_point,safety_stock,below_reorder_point,below_safety_stock,stock_ratio
0,333,72,22,False,False,4.625000
1,236,67,14,False,False,3.522388
2,496,96,34,False,False,5.166667
3,109,34,13,False,False,3.205882
4,333,97,22,False,False,3.432990


In [53]:
inventory_features["below_reorder_point"].value_counts()

below_reorder_point
False    19307
True      7101
Name: count, dtype: int64

In [54]:
inventory_features["below_safety_stock"].value_counts()

below_safety_stock
False    21199
True      5209
Name: count, dtype: int64

### Inventory Feature Engineering Summary

Inventory snapshot data was enriched with product information to create inventory-related features. Indicators were generated to identify products below their reorder point and safety stock levels. Additionally, a stock ratio feature was calculated to measure inventory availability relative to the reorder threshold. These engineered features will support inventory risk assessment and replenishment planning.

## Final Model Dataset

The engineered features created in previous sections are combined into a single analysis-ready dataset. This dataset will be used for demand forecasting and inventory risk modeling in the next phases.

In [61]:
print(sales_df.shape)

(9932614, 17)


In [55]:
model_df = sales_df.merge(
    sku_features,
    on="sku_id",
    how="left",
    suffixes=("", "_sku")
)

In [62]:
temp = sales_df.merge(
    sku_features,
    on="sku_id",
    how="left"
)

print(temp.shape)

(9932614, 27)


In [56]:
model_df = model_df.merge(
    customer_features,
    on="customer_id",
    how="left",
    suffixes=("", "_customer")
)

In [63]:
temp2 = temp.merge(
    customer_features,
    on="customer_id",
    how="left"
)

print(temp2.shape)

(9932614, 38)


In [57]:
model_df = model_df.merge(
    store_features,
    on="store_id",
    how="left",
    suffixes=("", "_store")
)

In [64]:
temp3 = temp2.merge(
    store_features,
    on="store_id",
    how="left"
)

print(temp3.shape)

(9932614, 46)


In [58]:
model_df = model_df.merge(
    sales_with_promotions[
        ["receipt_id", "promotion_used"]
    ],
    on="receipt_id",
    how="left"
)

In [65]:
temp4 = temp3.merge(
    sales_with_promotions[
        ["receipt_id", "promotion_used"]
    ],
    on="receipt_id",
    how="left"
)

print(temp4.shape)

(24944920, 47)


In [66]:
sales_with_promotions["receipt_id"].duplicated().sum()

np.int64(4764750)

In [67]:
promotion_lookup = (
    sales_with_promotions[
        ["receipt_id", "promotion_used"]
    ]
    .drop_duplicates(subset="receipt_id")
)

promotion_lookup.shape

(5167864, 2)

In [68]:
model_df = sales_df.copy()

In [69]:
model_df = model_df.merge(
    sku_features,
    on="sku_id",
    how="left",
    suffixes=("", "_sku")
)

In [70]:
model_df = model_df.merge(
    customer_features,
    on="customer_id",
    how="left",
    suffixes=("", "_customer")
)

In [71]:
model_df = model_df.merge(
    store_features,
    on="store_id",
    how="left",
    suffixes=("", "_store")
)

In [72]:
model_df = model_df.merge(
    promotion_lookup,
    on="receipt_id",
    how="left"
)

In [73]:
print("Sales DF:", sales_df.shape)
print("Model DF:", model_df.shape)

Sales DF: (9932614, 17)
Model DF: (9932614, 47)


In [74]:
sku_features["sku_id"].duplicated().sum()

np.int64(0)

In [75]:
customer_features["customer_id"].duplicated().sum()

np.int64(0)

In [76]:
store_features["store_id"].duplicated().sum()

np.int64(0)

In [77]:
promotion_lookup["receipt_id"].duplicated().sum()

np.int64(0)

In [79]:
model_df.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,registration_date,total_sales_store,total_quantity_store,total_transactions_store,average_order_value,store_name,city_store,store_type,opening_date,promotion_used
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,...,2020-09-29,2.575954e+08,443525,235843,1092.232527,Lahore Convenience Store #16,Lahore,Convenience Store,2016-06-07,False
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,...,2020-09-29,2.575954e+08,443525,235843,1092.232527,Lahore Convenience Store #16,Lahore,Convenience Store,2016-06-07,False
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,...,2025-10-12,2.574381e+08,442257,234792,1096.451818,Islamabad Convenience Store #15,Islamabad,Convenience Store,2020-05-11,False
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,...,2025-10-12,2.574381e+08,442257,234792,1096.451818,Islamabad Convenience Store #15,Islamabad,Convenience Store,2020-05-11,False
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,...,2022-07-10,5.140385e+08,886423,471308,1090.663659,Multan Hypermarket #20,Multan,Hypermarket,2017-06-30,False


In [80]:
model_df.shape

(9932614, 47)

In [81]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9932614 entries, 0 to 9932613
Data columns (total 47 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   date                         datetime64[us]
 1   receipt_id                   str           
 2   store_id                     str           
 3   sku_id                       str           
 4   customer_id                  str           
 5   quantity                     int64         
 6   unit_price                   float64       
 7   total_value                  float64       
 8   channel                      str           
 9   discount_pct                 float64       
 10  promo_id                     str           
 11  year                         int32         
 12  month                        int32         
 13  quarter                      int32         
 14  week                         int64         
 15  day_of_week                  str           
 16  is_weekend 

In [82]:
model_df.isnull().sum().sort_values(ascending=False)

promo_id                       7845164
receipt_id                           0
date                                 0
sku_id                               0
customer_id                          0
quantity                             0
store_id                             0
unit_price                           0
total_value                          0
channel                              0
discount_pct                         0
year                                 0
month                                0
quarter                              0
week                                 0
day_of_week                          0
is_weekend                           0
total_quantity                       0
total_sales                          0
average_price                        0
total_transactions                   0
sku_name                             0
category                             0
subcategory                          0
unit_price_sku                       0
cost_price               

In [83]:
model_df.duplicated().sum()

np.int64(0)

In [84]:
model_df.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,registration_date,total_sales_store,total_quantity_store,total_transactions_store,average_order_value,store_name,city_store,store_type,opening_date,promotion_used
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,...,2020-09-29,2.575954e+08,443525,235843,1092.232527,Lahore Convenience Store #16,Lahore,Convenience Store,2016-06-07,False
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,...,2020-09-29,2.575954e+08,443525,235843,1092.232527,Lahore Convenience Store #16,Lahore,Convenience Store,2016-06-07,False
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,...,2025-10-12,2.574381e+08,442257,234792,1096.451818,Islamabad Convenience Store #15,Islamabad,Convenience Store,2020-05-11,False
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,...,2025-10-12,2.574381e+08,442257,234792,1096.451818,Islamabad Convenience Store #15,Islamabad,Convenience Store,2020-05-11,False
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,...,2022-07-10,5.140385e+08,886423,471308,1090.663659,Multan Hypermarket #20,Multan,Hypermarket,2017-06-30,False


In [85]:
model_df.to_csv(
    "../data/processed/model_dataset.csv",
    index=False
)

print("Model dataset saved successfully!")

Model dataset saved successfully!
